In [1]:
# Cell 1: Imports and Main Execution
import logging
from pathlib import Path  # For path manipulation
import os # for cpu count
from nlp_utils import (
    clear_memory,
    init_worker,
    ner_spacy,
    ner_nltk,
    sentiment_spacy,
    sentiment_nltk,
    speed_spacy,
    speed_nltk,
    dialogue_detection_spacy,
    dialogue_detection_nltk,
    benchmark_performance,
    _process_chunk_or_sample,
    process_sample_in_chunks,
    merge_chunk_results,
    load_samples,
    create_ground_truth,
    load_ground_truth,
    analyze_and_save_results,
    create_visualizations,
    validate_against_ground_truth,
    remove_temp_files,
    close_database_connections,
    release_resources
)

# Set up logging to console (in addition to the file handler in nlp_utils)
logger = logging.getLogger("nlp_benchmark") # Get the SAME logger
ch = logging.StreamHandler()
formatter = logging.Formatter(
    "%(asctime)s - %(processName)s - %(levelname)s - %(message)s"
)
ch.setFormatter(formatter)
logger.addHandler(ch)

# Constants - Keep these here, too!
CORPUS_DIR = "../data/corpus"
GROUND_TRUTH_DIR = "../data/ground_truth"
RESULTS_DIR = "../results"
LOG_FILE = "benchmark.log" # This is already defined, but ok to do it again
CHUNK_SIZE = 75000
OVERLAP = 1000
MAX_WORKERS = 1

if __name__ == '__main__':
    # --- Create dummy data and ground truth if no corpus exists ---
    corpus_path = Path(CORPUS_DIR)
    if not corpus_path.exists() or not any(corpus_path.glob('*.txt')):
        logger.warning(f"No corpus found at {CORPUS_DIR}. Creating dummy data.")
        os.makedirs(CORPUS_DIR, exist_ok=True)
        os.makedirs(GROUND_TRUTH_DIR, exist_ok=True)

        dummy_samples = {
            "sample1.txt": 'This is a great movie! I absolutely loved it. "But the ending was so sad," she said.',
            "sample2.txt": "John Smith went to London. He met Jane Doe.  The food was terrible.",
        }

        for filename, text in dummy_samples.items():
            with open(os.path.join(CORPUS_DIR, filename), "w", encoding="utf-8") as f:
                f.write(text)

        # Create corresponding ground truth (adapt as needed)
        create_ground_truth(
            "category/sample1.txt",
            dummy_samples["sample1.txt"],
            dialogues=["But the ending was so sad,"],
            characters=[],
            emotions=[("This is a great movie!", 0.8), ("I absolutely loved it.", 0.9), ("But the ending was so sad,", -0.7)],
        )
        create_ground_truth(
            "category/sample2.txt",
            dummy_samples["sample2.txt"],
            dialogues=[],
            characters=["John Smith", "Jane Doe"],
            emotions=[("The food was terrible.", -0.75)],
        )
    # --- Load samples from disk ---
    samples = load_samples(directory=CORPUS_DIR)

    # Display sample information
    logger.info(f"Loaded {len(samples)} samples:")
    for sample_id, sample_text in samples.items():
        logger.info(f"  - {sample_id} ({len(sample_text)} chars)")

    # --- Process the samples ---
    all_results = process_multiple_samples(samples)  # Call the function

    # --- Load ground truth ---
    all_ground_truth = {
        sample_id: load_ground_truth(sample_id) for sample_id in samples
    }

    # --- Analyze and save results ---
    results_df = analyze_and_save_results(all_results, samples)

    # --- Validate Results and Visualize ---
    if results_df is not None and not results_df.empty:
        validation_df = validate_against_ground_truth(results_df, samples)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        create_visualizations(results_df, Path(RESULTS_DIR), timestamp)  # Use Path
    else:
        logger.warning("No results to validate or visualize.")

    # --- Cleanup ---
    remove_temp_files()
    close_database_connections()
    release_resources()

    logger.info("Benchmark run complete. Cleanup performed.")

NameError: name 'results_df' is not defined